### Setup script for CIFAR-10


In [3]:
import numpy as np
import pickle
import tarfile
from pathlib import Path

# Navigate 2 levels up from 'examples/pure_grit/' to the repository root
REPO_ROOT = Path(__file__).resolve().parents[2] if "__file__" in locals() else Path.cwd().parents[1]
DATA_PATH = REPO_ROOT / "data" / "cifar-10-python.tar.gz"

print(REPO_ROOT)

print(f"\n{DATA_PATH}")

/app

/app/data/cifar-10-python.tar.gz


When extracted, we expand a directory containing multiple pickle objects that split across 5 training sets, a testing set, a readme.html file, as well as a .meta file. 

We can pass in a function that maps a path (we define this), and a batch name(this is updated dynamically because there's 5 training and 1 test batch). 

#### Python `tarfile`:

We can use `tarfile.open` to read a compressed `tar` file which yields an index of file metadata records which are known as `TarInfo`. We auto close it when a *book* is completed. 

Now, apparently, rather then unpacking the files onto disk, we can use the line `f = tar.extractfile(member)` which returns a standard Python binary stream object `BufferedReader`/`GzipFile` wrapper. So we can read from ram instead...

#### Python `pickle`:

We convert Python object hierarchies into byte streams and then unpack them into python objects. 

In [4]:
def load_cifar10_all(tar_path):
    """
    Decompresses and extracts all CIFAR-10 training and testing batches 
    directly into Python lists/NumPy arrays without writing to disk.
    
    Returns:
        train_images (list): 50,000 raw 1D image byte vectors
        train_labels (list): 50,000 integer labels (0-9)
        test_images (list):  10,000 raw 1D image byte vectors
        test_labels (list):  10,000 integer labels (0-9)
    """
    train_images = []
    train_labels = []
    
    test_images = []
    test_labels = []

    with tarfile.open(tar_path, "r:gz") as tar:
        

        for i in range(1, 6):
            batch_name = f"cifar-10-batches-py/data_batch_{i}"
            
            member = tar.getmember(batch_name)
            f = tar.extractfile(member)
            batch_dict = pickle.load(f, encoding="bytes")
            

            train_images.append(batch_dict[b"data"])
            train_labels.extend(batch_dict[b"labels"])


        test_member = tar.getmember("cifar-10-batches-py/test_batch")
        test_f = tar.extractfile(test_member)
        test_dict = pickle.load(test_f, encoding="bytes")
        
        test_images.append(test_dict[b"data"])
        test_labels.extend(test_dict[b"labels"])



    train_images = np.vstack(train_images)
    test_images = np.vstack(test_images)

    return train_images, train_labels, test_images, test_labels

In [8]:
X_train, y_train, X_test, y_test = load_cifar10_all(DATA_PATH)

print(f"X_train shape: {X_train.shape}")  # Output: (50000, 3072)
print(f"y_train count: {len(y_train)}")     # Output: 50000
print(f"X_test shape:  {X_test.shape}")   # Output: (10000, 3072)
print(f"y_test count:  {len(y_test)}")      # Output: 10000

X_train shape: (50000, 3072)
y_train count: 50000
X_test shape:  (10000, 3072)
y_test count:  10000


### How to Reshape the axes...

We know that each input image contains 3072 pixel elements, where the height and width dimensions are both "32x32 colour images" meaning that 
$H=32\quad W=32 \quad C=3$. That means, doing $3 \times 32 \times 32 = 3072$.

Since we already know the sample size of X_train and sample size of X_test, we can say that 

$$
\text{X\_train} = N\times CHW = (50000, 3,\times 32\times 32)
$$

$$
\text{X\_test} = N\times CHW = (10000, 3,\times 32\times 32)
$$

Goal is to reshape the axes to become
$$
NHWC = (N, 32, 32, 3)
$$

First, we need to reshape the axes into the $NCHW$ tensor, then we can transpose the disjoint axes to have a true $NHWC$ layout. 

In [10]:
train_nhwc = X_train.reshape(-1, 3, 32, 32)
test_nchw = X_test.reshape(-1, 3, 32, 32)

images_nhwc_train = np.transpose(train_nhwc, axes=(0, 2, 3, 1))
images_nhwc_test = np.transpose(test_nchw, axes=(0, 2, 3, 1))

#This creates a non-contiguous view in memory, so we'll account for this
images_nhwc_train = np.ascontiguousarray(images_nhwc_train)
images_nhwc_test = np.ascontiguousarray(images_nhwc_test)

In [11]:
print(images_nhwc_train.shape)                  # (50000, 32, 32, 3)
print(images_nhwc_train.flags['C_CONTIGUOUS']) # True

(50000, 32, 32, 3)
True


#### Saving our test set for future use

We now need a way to load our data into disk. To keep this repository lightweight, we can use `numpy.savez` or `numpy.savez_compressed` Which lets us provide arrays as keyword arguments to store them under their corresponding name.
  
`  
numpy.savez(file, *args, allow_pickle=True, **kwds)
`
  
On the API reference:

`allow_pickle`: Allow saving object arrays using Python pickles, but for security reasons I will disallow python pickle so we'll set this boolean to false. 

`kwds`: These are our arrays to save to the file, with each array being saved to the output file but each having a corresponding keyword name

In [14]:
from pathlib import Path
import numpy as np

SAVE_TRAIN_TO = REPO_ROOT / "data" / "cifar-10" / "cifar-10_train.npz"

# If you do not yet have a data folder or a cifar-10 folder, this makes
# it for you
SAVE_TRAIN_TO.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    SAVE_TRAIN_TO,
    X_train=images_nhwc_train,
    y_train=y_train
)
SAVE_TEST_TO = REPO_ROOT / "data" / "cifar-10" / "cifar-10_test.npz"
np.savez_compressed(
    SAVE_TEST_TO,
    X_test=images_nhwc_test,
    y_test=y_test
)

In [17]:
# Final check to see that it worked
with np.load(SAVE_TRAIN_TO, allow_pickle=False) as data:
    X_train = data["X_train"]
    y_train = data["y_train"]

print(X_train.shape)

with np.load(SAVE_TEST_TO, allow_pickle=False) as data:
    X_test = data["X_test"]
    y_test = data["y_test"]

print(X_test.shape)



(50000, 32, 32, 3)
(10000, 32, 32, 3)


In [ ]:
import numpy as np
import cupy as cp

# Both reference the exact same underlying numpy.dtype object
# Interesting quirk 
print(cp.float32 is np.float32)  # Output: True
print(cp.dtype('float32') == np.dtype('float32'))  # Output: True

True
True
